In [1]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

for filename in uploaded:
    with zipfile.ZipFile(filename, "r") as z:
        z.extractall("/content")

os.chdir("/content/lab10")

!pip install -q "torch>=2.4,<3" "transformers==4.48.3" "datasets==3.2.0" "matplotlib>=3.7,<4" "pandas>=2,<3"

Saving lab10.zip to lab10.zip
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.9.0 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from transformers import AutoTokenizer
import pandas as pd
from qa_model import align, features

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased", use_fast=True)
question = "What is the capital of France?"
context = "The capital of France is Paris."
answer = "Paris"
answer_start = context.index(answer)
answer_end = answer_start + len(answer)
encoded = tokenizer(question, context, return_offsets_mapping=True)
offsets = encoded["offset_mapping"]
sequence_ids = encoded.sequence_ids()
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
pd.DataFrame({"token_index": range(len(tokens)), "token": tokens, "input_ids": encoded["input_ids"], "attention_mask": encoded["attention_mask"], "sequence_id": sequence_ids, "offset_mapping": offsets})

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

,token_index,token,input_ids,attention_mask,sequence_id,offset_mapping
0,0,[CLS],101,1,NaN,"(0, 0)"
1,1,what,2054,1,0.0,"(0, 4)"
2,2,is,2003,1,0.0,"(5, 7)"
3,3,the,1996,1,0.0,"(8, 11)"
4,4,capital,3007,1,0.0,"(12, 19)"
5,5,of,1997,1,0.0,"(20, 22)"
6,6,france,2605,1,0.0,"(23, 29)"
7,7,?,1029,1,0.0,"(29, 30)"
8,8,[SEP],102,1,NaN,"(0, 0)"
9,9,the,1996,1,1.0,"(0, 3)"


Input IDs biểu diễn token trong vocabulary. Attention mask xác định token thực. DistilBERT không sử dụng token type IDs. Sequence ID 0 thuộc question, 1 thuộc context và None thuộc special token.

In [9]:
{"input_keys": list(encoded.keys()), "uses_token_type_ids": "token_type_ids" in encoded, "character_start": answer_start, "character_end_exclusive": answer_end}

{'input_keys': ['input_ids', 'attention_mask', 'offset_mapping'],
 'uses_token_type_ids': False,
 'character_start': 25,
 'character_end_exclusive': 30}

In [3]:
start, end = align(offsets, sequence_ids, answer_start, answer_end)
assert (start, end) == (14, 14)
assert context[offsets[start][0]:offsets[end][1]] == answer
{"start_token": start, "end_token": end, "answer": context[offsets[start][0]:offsets[end][1]]}

{'start_token': 14, 'end_token': 14, 'answer': 'Paris'}

## Answer bắt đầu hoặc kết thúc giữa token

In [5]:
cases = ["Paris", "aris", "Pari", "capital of France"]
records = []
for text in cases:
    a = context.index(text)
    b = a + len(text)
    s, e = align(offsets, sequence_ids, a, b)
    records.append({"gold": text, "char_start": a, "char_end": b, "token_start": s, "token_end": e, "reconstructed": context[offsets[s][0]:offsets[e][1]]})
pd.DataFrame(records)

,gold,char_start,char_end,token_start,token_end,reconstructed
0,Paris,25,30,14,14,Paris
1,aris,26,30,14,14,Paris
2,Pari,25,29,14,14,Paris
3,capital of France,4,21,10,12,capital of France


Khi biên nằm giữa token, target được căn chỉnh tới token chứa biên đó. Span khôi phục có thể rộng hơn gold vì QA head không dự đoán biên ở cấp ký tự.

## Sliding window và overflow mapping

In [6]:
long_context = "The museum has many visitors. " * 90 + context + " The park is open every day." * 60
long_start = long_context.index(answer)
inputs = tokenizer([question, question], [context, long_context], truncation="only_second", max_length=384, stride=128, return_overflowing_tokens=True, return_offsets_mapping=True)
summary = []
for i, sample in enumerate(inputs["overflow_to_sample_mapping"]):
    a = answer_start if sample == 0 else long_start
    cls = inputs["input_ids"][i].index(tokenizer.cls_token_id)
    s, e = align(inputs["offset_mapping"][i], inputs.sequence_ids(i), a, a + len(answer), cls)
    summary.append({"window": i, "sample": sample, "length": len(inputs["input_ids"][i]), "start_token": s, "end_token": e, "contains_answer": s != cls})
pd.DataFrame(summary)

,window,sample,length,start_token,end_token,contains_answer
0,0,0,17,14,14,True
1,1,1,384,0,0,False
2,2,1,384,308,308,True
3,3,1,384,62,62,True
4,4,1,239,0,0,False


Overflow to sample mapping xác định mỗi window thuộc sample nào. Window không chứa toàn bộ answer có target tại CLS.

## Answer bị truncation và nằm ngoài window

In [7]:
synthetic_offsets = [(0, 0), (0, 3), (4, 11), (12, 14), (15, 21), (0, 0)]
synthetic_ids = [None, 1, 1, 1, 1, None]
checks = [("inside", 4, 21), ("partial truncation", 15, 30), ("outside", 25, 30), ("inside token", 5, 10)]
results = [{"case": name, "target": align(synthetic_offsets, synthetic_ids, a, b)} for name, a, b in checks]
assert results[0]["target"] == (2, 4)
assert results[1]["target"] == (0, 0)
assert results[2]["target"] == (0, 0)
assert results[3]["target"] == (2, 2)
pd.DataFrame(results)

,case,target
0,inside,"(2, 4)"
1,partial truncation,"(0, 0)"
2,outside,"(0, 0)"
3,inside token,"(2, 2)"


In [8]:
examples = [{"question": question, "context": long_context, "answers": {"answer_start": [long_start], "text": [answer]}}]
rows = features(tokenizer, examples, 384, 128)
assert any(r["start_positions"] != 0 for r in rows)
assert any(r["start_positions"] == 0 for r in rows)
pd.DataFrame([{"window": i, "start": r["start_positions"], "end": r["end_positions"]} for i, r in enumerate(rows)])

,window,start,end
0,0,0,0
1,1,308,308
2,2,62,62
3,3,0,0


In [10]:
%cd /content/lab10

!zip -q /content/lab10_results.zip *.md *.csv *.png *.json *.ipynb

from google.colab import files
files.download("/content/lab10_results.zip")

/content


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>